# Learning LLM Engineering: Mastering Tokenizers

Tokenizers are the first and last step in every LLM pipeline. They convert human-readable text into numerical Token IDs that models can process, and vice-versa.

In [ ]:
# Setting up my environment with the specific versions of datasets and transformers I'll be using.
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [ ]:
# Bringing in the tools I need to handle authentication and model interaction.
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

In [ ]:
# Connecting to Hugging Face and confirming I have GPU access, which is crucial for efficient model testing.
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

gpu_info = !nvidia-smi
if 'failed' not in '\n'.join(gpu_info):
    print("I have a GPU active and ready for inference.")

In [ ]:
# Initializing the Llama 3.1 tokenizer. I need to understand how this specific model views text.
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)

In [ ]:
# Converting my test string into numerical IDs. This is the 'language' the model actually speaks.
text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
display(tokens)

In [ ]:
# Checking the token-to-word ratio. Understanding that one word doesn't always equal one token is key for cost and context window management.
print(f"Chars: {len(text)} | Words: {len(text.split())} | Tokens: {len(tokens)}")

In [ ]:
# Decoding the IDs back to text to verify the round-trip process and see any hidden special characters.
tokenizer.decode(tokens)

In [ ]:
# Looking at how the tokenizer actually fragments the words into sub-units.
tokenizer.batch_decode(tokens)

In [ ]:
# Inspecting the special tokens (like BOS/EOS) that help the model understand structure.
tokenizer.get_added_vocab()

In [ ]:
# Checking the total vocabulary size of the Llama 3.1 tokenizer.
len(tokenizer.vocab)

## My notes on Chat Templates

I've learned that 'Instruct' models need specific formatting to distinguish between system instructions and user input. `apply_chat_template` is the tool I'll use to automate this formatting so I don't have to manually write complex tags.

In [ ]:
# Load an 'Instruct' version which uses specific chat formatting
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

In [ ]:
# Standardizing multi-turn conversation into a single string
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

## The Core Realization

I used to think LLMs processed Python lists or dictionaries directly. Now I see the pipeline clearly:
1. My message list is turned into a single formatted string.
2. That string is broken into sub-word tokens.
3. Tokens are mapped to IDs.

**The input is always just a sequence of numbers.**

## Comparative Analysis of Tokenizers

Different families (Microsoft, DeepSeek, Qwen) use different vocabularies and splitting logic. As an engineer, choosing the right tokenizer affects latency and cost.

In [ ]:
# Keeping track of the different model paths I want to compare.
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [ ]:
# Compare how Llama and Phi-4 tokenize the same sentence
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

print("Llama IDs:", tokenizer.encode(text))
print("Phi-4 IDs:", phi4_tokenizer.encode(text))

In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
# Comparing how different tokenizers (Llama, Phi, DeepSeek) encode the same string.
# I can see that different models use different ID mappings for the same words.
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

print("Llama IDs:", tokenizer.encode(text))
print("Phi-4 IDs:", phi4_tokenizer.encode(text))
print("DeepSeek IDs:", deepseek_tokenizer.encode(text))

In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
# Experimenting with a coder-specific tokenizer to see how it handles Python syntax like indentation and keywords.
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code)
for t in tokens:
    print(f"{t} -> '{qwen_tokenizer.decode(t)}'")